In [ ]:
import torch, os

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.count(os.sep) - 2
    if depth <= 4:
        print("  " * depth + os.path.basename(root) + "/", f"({len(files)} files)")

In [ ]:
# Data understanding(EDA)

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset")

rows = []
for split in ["Training", "Testing"]:
    for class_dir in sorted((DATA_DIR / split).iterdir()):
        for img in class_dir.iterdir():
            rows.append({"split": split, "label": class_dir.name, "path": str(img)})

df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/metadata.csv", index=False)

print("Total images:", len(df))
pd.crosstab(df["label"], df["split"], margins=True)

In [ ]:
import matplotlib.pyplot as plt

counts = df.groupby(["label", "split"]).size().unstack()
counts.plot(kind="bar", figsize=(8, 4), color=["#0D1F4E", "#A51C24"])
plt.title("Images per class")
plt.ylabel("Number of images")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:


#

In [ ]:
from PIL import Image

classes = sorted(df["label"].unique())
fig, axes = plt.subplots(len(classes), 5, figsize=(12, 10))

for i, cls in enumerate(classes):
    samples = df[(df["label"] == cls) & (df["split"] == "Training")].sample(5, random_state=42)
    for j, path in enumerate(samples["path"]):
        axes[i, j].imshow(Image.open(path), cmap="gray")
        axes[i, j].axis("off")
    axes[i, 0].set_title(cls, loc="left", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
#image size and the colors modes 

In [ ]:
sizes = []
for path in df["path"]:
    with Image.open(path) as im:
        sizes.append({"width": im.width, "height": im.height, "mode": im.mode})

info = pd.DataFrame(sizes)
print("Colour modes:\n", info["mode"].value_counts(), "\n")
print("Most common sizes:\n", info.value_counts(["width", "height"]).head(10), "\n")
print(info.describe())

In [ ]:
#chek for duplicates 

In [ ]:
!pip install -q imagehash
import imagehash
from tqdm.auto import tqdm

hashes = []
for path in tqdm(df["path"], desc="Hashing"):
    with Image.open(path) as im:
        hashes.append(str(imagehash.phash(im.convert("L"))))
df["phash"] = hashes

dups = df[df.duplicated("phash", keep=False)]
print("Images with at least one duplicate:", len(dups))
print("Duplicate groups:", dups["phash"].nunique())

train_h = set(df.loc[df["split"] == "Training", "phash"])
test_h = set(df.loc[df["split"] == "Testing", "phash"])
leaked = df[(df["split"] == "Testing") & (df["phash"].isin(train_h & test_h))]
print("Test images that also appear in Training (LEAKAGE):", len(leaked))

label_conflicts = df.groupby("phash")["label"].nunique()
print("Duplicate groups with DIFFERENT labels:", (label_conflicts > 1).sum())

df.to_csv("/kaggle/working/metadata.csv", index=False)

In [ ]:
#look for duplicate 

In [ ]:
groups = dups.groupby("phash")
fig, axes = plt.subplots(4, 3, figsize=(9, 12))
for row, (h, g) in zip(axes, list(groups)[:4]):
    for ax in row:
        ax.axis("off")
    for ax, (_, r) in zip(row, g.head(3).iterrows()):
        ax.imshow(Image.open(r["path"]), cmap="gray")
        ax.set_title(f'{r["split"]} / {r["label"]}', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
#data cleaning and splitting 

In [ ]:
conflict_hashes = label_conflicts[label_conflicts > 1].index
clean = df[~df["phash"].isin(conflict_hashes)].copy()

test_df = clean[clean["split"] == "Testing"].drop_duplicates("phash")

train_df = clean[
    (clean["split"] == "Training") & (~clean["phash"].isin(test_df["phash"]))
].drop_duplicates("phash")

print("Removed for conflicting labels:", len(df) - len(clean))
print("Clean Testing images:", len(test_df))
print("Clean Training images:", len(train_df))
print("Total removed:", len(df) - len(test_df) - len(train_df))

In [ ]:
#validation set 

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    train_df, test_size=0.15, stratify=train_df["label"], random_state=42
)

train_df = train_df.assign(set="train")
val_df = val_df.assign(set="val")
test_df = test_df.assign(set="test")

final = pd.concat([train_df, val_df, test_df], ignore_index=True)
final.to_csv("/kaggle/working/clean_metadata.csv", index=False)

pd.crosstab(final["label"], final["set"], margins=True)

In [ ]:
#image preprocessing 

In [ ]:
import cv2
import numpy as np

IMG_SIZE = 256

def crop_brain(img, pad=10):
    """Remove the black background around the brain."""
    gray = np.array(img.convert("L"))
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, mask = cv2.threshold(blur, 20, 255, cv2.THRESH_BINARY)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    rgb = img.convert("RGB")
    if not contours:
        return rgb
    x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
    H, W = gray.shape
    return rgb.crop((max(x - pad, 0), max(y - pad, 0), min(x + w + pad, W), min(y + h + pad, H)))

def resize_keep_shape(img, size=IMG_SIZE):
    """Resize without stretching, then place on a black square."""
    scale = size / max(img.size)
    new = img.resize((round(img.width * scale), round(img.height * scale)), Image.BICUBIC)
    canvas = Image.new("RGB", (size, size))
    canvas.paste(new, ((size - new.width) // 2, (size - new.height) // 2))
    return canvas

def preprocess(path):
    with Image.open(path) as im:
        return resize_keep_shape(crop_brain(im))

samples = final.groupby("label").sample(2, random_state=1)
fig, axes = plt.subplots(2, len(samples), figsize=(16, 5))
for k, (_, r) in enumerate(samples.iterrows()):
    axes[0, k].imshow(Image.open(r["path"]), cmap="gray"); axes[0, k].set_title(f'{r["label"]}\noriginal', fontsize=9)
    axes[1, k].imshow(preprocess(r["path"])); axes[1, k].set_title("processed", fontsize=9)
    axes[0, k].axis("off"); axes[1, k].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
#process all images and save them 

In [ ]:
from joblib import Parallel, delayed
import os

out_dir = Path("/kaggle/working/processed")

def process_one(i, src, set_name, label):
    dest = out_dir / set_name / label / f"{i}.png"
    if not dest.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        preprocess(src).save(dest, compress_level=1)
    return str(dest)

new_paths = Parallel(n_jobs=os.cpu_count())(
    delayed(process_one)(i, r["path"], r["set"], r["label"])
    for i, r in tqdm(final.iterrows(), total=len(final), desc="Preprocessing")
)

final["proc_path"] = new_paths
final.to_csv("/kaggle/working/clean_metadata.csv", index=False)
print("Saved", len(final), "images to", out_dir)

In [ ]:
#quick check for the error 

In [ ]:
for s in ["train", "val", "test"]:
    print(s, sum(1 for _ in (out_dir / s).rglob("*.png")))

In [ ]:
#dataset and dataloader 

In [ ]:
!pip install -q timm
import torch, torch.nn as nn, timm, time
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from sklearn.metrics import f1_score

device = "cuda"
CLASSES = sorted(final["label"].unique())
cls2idx = {c: i for i, c in enumerate(CLASSES)}
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_tf = T.Compose([
    T.RandomResizedCrop(224, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize(MEAN, STD)])

class MRIDataset(Dataset):
    def __init__(self, frame, tf):
        self.paths = frame["proc_path"].tolist()
        self.labels = frame["label"].map(cls2idx).tolist()
        self.tf = tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert("RGB")), self.labels[i]

tr = final[final["set"] == "train"]
va = final[final["set"] == "val"]
train_dl = DataLoader(MRIDataset(tr, train_tf), batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
val_dl = DataLoader(MRIDataset(va, eval_tf), batch_size=64, num_workers=4, pin_memory=True)

counts = tr["label"].value_counts().reindex(CLASSES).values
class_weights = torch.tensor(len(tr) / (len(CLASSES) * counts), dtype=torch.float, device=device)
print("Classes:", CLASSES)
print("Class weights:", dict(zip(CLASSES, class_weights.cpu().numpy().round(2))))

In [ ]:
#model,loss and training setting 

In [ ]:
EPOCHS = 15
model = timm.create_model("efficientnet_b0.ra_in1k", pretrained=True, num_classes=len(CLASSES)).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_dl))
scaler = torch.amp.GradScaler("cuda")

print("Parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "million")

In [ ]:
#

In [ ]:
broken = []
for i, r in tqdm(final.iterrows(), total=len(final), desc="Checking files"):
    try:
        with Image.open(r["proc_path"]) as im:
            im.load()
    except Exception:
        broken.append(i)

print("Broken files found:", len(broken))
for i in broken:
    r = final.loc[i]
    preprocess(r["path"]).save(r["proc_path"], compress_level=1)
print("Re-created:", len(broken))

In [ ]:
#the training loop 

In [ ]:
def run_epoch(loader, train=True):
    model.train(train)
    total_loss, preds, targets = 0.0, [], []
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.set_grad_enabled(train), torch.amp.autocast("cuda"):
            out = model(x)
            loss = criterion(out, y)
        if train:
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
        total_loss += loss.item() * len(y)
        preds += out.argmax(1).tolist()
        targets += y.tolist()
    acc = np.mean(np.array(preds) == np.array(targets))
    f1 = f1_score(targets, preds, average="macro")
    return total_loss / len(loader.dataset), acc, f1

history, best_f1, patience, bad = [], 0, 4, 0
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tl, ta, tf1 = run_epoch(train_dl, train=True)
    vl, vacc, vf1 = run_epoch(val_dl, train=False)
    history.append(dict(epoch=epoch, train_loss=tl, val_loss=vl, train_acc=ta, val_acc=vacc, val_f1=vf1))
    flag = ""
    if vf1 > best_f1:
        best_f1, bad = vf1, 0
        torch.save(model.state_dict(), "/kaggle/working/best_effnet_b0.pt")
        flag = "  <- best, saved"
    else:
        bad += 1
    print(f"Epoch {epoch:2d} | train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {vacc:.3f} F1 {vf1:.3f} | {time.time()-t0:.0f}s{flag}")
    if bad >= patience:
        print("Early stopping: no improvement for", patience, "epochs")
        break

print("Best validation macro-F1:", round(best_f1, 4))

In [ ]:
#learning curve 

In [ ]:
h = pd.DataFrame(history)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
h.plot(x="epoch", y=["train_loss", "val_loss"], ax=ax[0], title="Loss", marker="o")
h.plot(x="epoch", y=["train_acc", "val_acc", "val_f1"], ax=ax[1], title="Accuracy / macro-F1", marker="o")
plt.tight_layout(); plt.show()

In [ ]:
#overall validation after the training 

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

@torch.no_grad()
def predict(loader, weights_path, arch):
    m = timm.create_model(arch, pretrained=False, num_classes=len(CLASSES)).to(device)
    m.load_state_dict(torch.load(weights_path, map_location=device))
    m.eval()
    preds, targets = [], []
    for x, y in loader:
        with torch.amp.autocast("cuda"):
            preds += m(x.to(device)).argmax(1).tolist()
        targets += y.tolist()
    return np.array(targets), np.array(preds)

y_true, y_pred = predict(val_dl, "/kaggle/working/best_effnet_b0.pt", "efficientnet_b0.ra_in1k")
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))

ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=CLASSES).plot(cmap="Blues")
plt.title("EfficientNet-B0 — validation"); plt.show()

In [ ]:
#training a stronger model and compare

In [ ]:
def fit(arch, lr, save_path, epochs=15, patience=4):
    global model, optimizer, scheduler, scaler
    model = timm.create_model(arch, pretrained=True, num_classes=len(CLASSES)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs * len(train_dl))
    scaler = torch.amp.GradScaler("cuda")
    best, bad, hist = 0, 0, []
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tl, ta, _ = run_epoch(train_dl, train=True)
        vl, vacc, vf1 = run_epoch(val_dl, train=False)
        hist.append(dict(epoch=epoch, train_loss=tl, val_loss=vl, val_acc=vacc, val_f1=vf1))
        flag = ""
        if vf1 > best:
            best, bad = vf1, 0
            torch.save(model.state_dict(), save_path)
            flag = "  <- best, saved"
        else:
            bad += 1
        print(f"Epoch {epoch:2d} | train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {vacc:.3f} F1 {vf1:.3f} | {time.time()-t0:.0f}s{flag}")
        if bad >= patience:
            print("Early stopping"); break
    print("Best validation macro-F1:", round(best, 4))
    return pd.DataFrame(hist)

hist_convnext = fit("convnext_tiny.fb_in22k_ft_in1k", lr=1e-4, save_path="/kaggle/working/best_convnext_tiny.pt")

In [ ]:
#compare everything that matters for magnetic resonance scanner 

In [ ]:
import os

y_true, y_pred_cx = predict(val_dl, "/kaggle/working/best_convnext_tiny.pt", "convnext_tiny.fb_in22k_ft_in1k")
print("ConvNeXt-Tiny — validation")
print(classification_report(y_true, y_pred_cx, target_names=CLASSES, digits=3))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred_cx), display_labels=CLASSES).plot(cmap="Blues")
plt.title("ConvNeXt-Tiny — validation"); plt.show()

def cpu_ms_per_image(arch, runs=20):
    m = timm.create_model(arch, pretrained=False, num_classes=len(CLASSES)).eval()
    x = torch.randn(1, 3, 224, 224)
    with torch.no_grad():
        for _ in range(3):
            m(x)
        t0 = time.time()
        for _ in range(runs):
            m(x)
    return (time.time() - t0) / runs * 1000

models = {
    "EfficientNet-B0": ("efficientnet_b0.ra_in1k", "/kaggle/working/best_effnet_b0.pt"),
    "ConvNeXt-Tiny": ("convnext_tiny.fb_in22k_ft_in1k", "/kaggle/working/best_convnext_tiny.pt"),
}
rows = []
for name, (arch, path) in models.items():
    yt, yp = predict(val_dl, path, arch)
    rows.append({
        "model": name,
        "val_macro_F1": round(f1_score(yt, yp, average="macro"), 4),
        "val_errors": int((yt != yp).sum()),
        "tumours_missed_as_notumor": int(((yt != cls2idx["notumor"]) & (yp == cls2idx["notumor"])).sum()),
        "file_size_MB": round(os.path.getsize(path) / 1e6, 1),
        "CPU_ms_per_image": round(cpu_ms_per_image(arch), 1),
    })
pd.DataFrame(rows)

In [ ]:
#final evaluation (test set evaluation )

In [ ]:
te = final[final["set"] == "test"]
test_dl = DataLoader(MRIDataset(te, eval_tf), batch_size=64, num_workers=4, pin_memory=True)

FINAL_ARCH = "convnext_tiny.fb_in22k_ft_in1k"
FINAL_WEIGHTS = "/kaggle/working/best_convnext_tiny.pt"

y_true_t, y_pred_t = predict(test_dl, FINAL_WEIGHTS, FINAL_ARCH)

print("FINAL TEST RESULTS — ConvNeXt-Tiny")
print(classification_report(y_true_t, y_pred_t, target_names=CLASSES, digits=3))
missed = int(((y_true_t != cls2idx["notumor"]) & (y_pred_t == cls2idx["notumor"])).sum())
print("Total test images:", len(y_true_t))
print("Total mistakes:", int((y_true_t != y_pred_t).sum()))
print("Tumours missed as 'notumor':", missed)

ConfusionMatrixDisplay(confusion_matrix(y_true_t, y_pred_t), display_labels=CLASSES).plot(cmap="Blues")
plt.title("ConvNeXt-Tiny — TEST set"); plt.show()

In [ ]:
#diagnosing 

In [ ]:
df_sz = df.assign(width=info["width"].values, height=info["height"].values, mode=info["mode"].values)
f2 = final.merge(df_sz[["path", "width", "height", "mode"]], on="path", how="left")
f2["is_512"] = (f2["width"] == 512) & (f2["height"] == 512)

print("Share of 512x512 images, by class and set:")
print(pd.crosstab(f2["label"], f2["set"], values=f2["is_512"], aggfunc="mean").round(2), "\n")

print("Colour mode by class and set:")
print(pd.crosstab([f2["label"], f2["set"]], f2["mode"]), "\n")

te_sz = f2[f2["set"] == "test"].reset_index(drop=True)
te_sz["pred"] = [CLASSES[p] for p in y_pred_t]
g = te_sz[te_sz["label"] == "glioma"]
print("Test gliomas — correct vs wrong, by image size group:")
print(pd.crosstab(g["is_512"], g["pred"] == "glioma", rownames=["is 512x512"], colnames=["correct"]))

In [ ]:
#re-spliting 

In [ ]:
f2["src"] = np.where(f2["is_512"], "512", "other")
f2["strata"] = f2["label"] + "_" + f2["src"]

trainval_new, test_new = train_test_split(f2, test_size=0.20, stratify=f2["strata"], random_state=42)
train_new, val_new = train_test_split(trainval_new, test_size=0.15, stratify=trainval_new["strata"], random_state=42)

train_new = train_new.assign(set="train")
val_new = val_new.assign(set="val")
test_new = test_new.assign(set="test")
final2 = pd.concat([train_new, val_new, test_new], ignore_index=True)
final2.to_csv("/kaggle/working/clean_metadata_v2.csv", index=False)

print(pd.crosstab([final2["label"], final2["src"]], final2["set"], margins=True))

In [ ]:
#shortcut augmentation then training 

In [ ]:
f2["src"] = np.where(f2["is_512"], "512", "other")
f2["strata"] = f2["label"] + "_" + f2["src"]

trainval_new, test_new = train_test_split(f2, test_size=0.20, stratify=f2["strata"], random_state=42)
train_new, val_new = train_test_split(trainval_new, test_size=0.15, stratify=trainval_new["strata"], random_state=42)

train_new = train_new.assign(set="train")
val_new = val_new.assign(set="val")
test_new = test_new.assign(set="test")
final2 = pd.concat([train_new, val_new, test_new], ignore_index=True)
final2.to_csv("/kaggle/working/clean_metadata_v2.csv", index=False)

print(pd.crosstab([final2["label"], final2["src"]], final2["set"], margins=True))

In [ ]:
#evaluation check 

In [ ]:
import random

class RandomLowRes:
    """Randomly lowers image resolution, so sharpness can't reveal the source."""
    def __init__(self, p=0.5, min_size=80, max_size=200):
        self.p, self.min_size, self.max_size = p, min_size, max_size
    def __call__(self, img):
        if random.random() < self.p:
            s = random.randint(self.min_size, self.max_size)
            return img.resize((s, s), Image.BILINEAR).resize(img.size, Image.BILINEAR)
        return img

train_tf2 = T.Compose([
    T.Grayscale(num_output_channels=3),
    RandomLowRes(p=0.5),
    T.RandomResizedCrop(224, scale=(0.75, 1.0), ratio=(0.9, 1.1)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 1.5))], p=0.3),
    T.ColorJitter(brightness=0.3, contrast=0.3),
    T.RandomAutocontrast(p=0.3),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])
eval_tf2 = T.Compose([
    T.Grayscale(num_output_channels=3),
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

tr2, va2, te2 = (final2[final2["set"] == s] for s in ["train", "val", "test"])
train_dl = DataLoader(MRIDataset(tr2, train_tf2), batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
val_dl = DataLoader(MRIDataset(va2, eval_tf2), batch_size=64, num_workers=4, pin_memory=True)
test_dl2 = DataLoader(MRIDataset(te2, eval_tf2), batch_size=64, num_workers=4, pin_memory=True)

counts2 = tr2["label"].value_counts().reindex(CLASSES).values
class_weights = torch.tensor(len(tr2) / (len(CLASSES) * counts2), dtype=torch.float, device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

hist_v2 = fit("convnext_tiny.fb_in22k_ft_in1k", lr=1e-4,
              save_path="/kaggle/working/best_convnext_tiny_v2.pt", epochs=20, patience=5)

In [ ]:
y_t2, y_p2 = predict(test_dl2, "/kaggle/working/best_convnext_tiny_v2.pt", "convnext_tiny.fb_in22k_ft_in1k")
print("ConvNeXt-Tiny v2 — NEW test set")
print(classification_report(y_t2, y_p2, target_names=CLASSES, digits=3))
print("Tumours missed as 'notumor':", int(((y_t2 != cls2idx["notumor"]) & (y_p2 == cls2idx["notumor"])).sum()))

res = te2.reset_index(drop=True).assign(correct=(y_t2 == y_p2))
print("\nAccuracy by class and source:")
print(res.pivot_table(index="label", columns="src", values="correct", aggfunc=["mean", "count"]).round(3))

In [ ]:
res["pred"] = [CLASSES[p] for p in y_p2]
bad = res[~res["correct"]]
print(bad[["label", "src", "pred"]].value_counts())

n = len(bad); cols = 4; rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(14, 3.6 * rows))
for ax in np.array(axes).ravel():
    ax.axis("off")
for ax, (_, r) in zip(np.array(axes).ravel(), bad.iterrows()):
    ax.imshow(Image.open(r["proc_path"]))
    ax.set_title(f'true: {r["label"]} ({r["src"]})\npred: {r["pred"]}', fontsize=9, color="#A51C24")
plt.tight_layout(); plt.show()